In [2]:

from __future__ import annotations

import operator
from pathlib import Path
from typing import TypedDict, List, Annotated, Literal,Optional

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_community.tools.tavily_search import TavilySearchResults

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# ====================== MODELS ======================
class Task(BaseModel):
    id: int
    title: str
    goal: str = Field(
        ..., 
        description="One sentence describing what the reader should be able to do/understand after this section."
    )
    bullets: List[str] = Field(
        ..., 
        min_length=3,
        max_length=5,
        description="3–5 concrete, non-overlapping subpoints to cover in this section."
    )
    target_words: int = Field(
        ..., 
        description="Target word count for this section (120–450)."
    )
    section_type: Literal[
        "intro", "core", "examples", "checklist", "common_mistakes", "conclusion"
    ] = Field(
        ..., 
        description="Use 'common_mistakes' exactly once in the plan."
    )


class Plan(BaseModel):
    blog_title: str
    audience: str = Field(..., description="Who this blog is for.")
    tone: str = Field(..., description="Writing tone (e.g., practical, crisp).")
    tasks: List[Task]


class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at: Optional[str] = None
    snippet: Optional[str] = None
    source: Optional[str] = None


class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal['closed_book', 'hybrid', 'open_book']
    queries: List[str] = Field(default_factory=list)


class EvidencePack(BaseModel):
    evidence: List[EvidenceItem] = Field(default_factory=list)


# ====================== STATE ======================
class State(TypedDict):
    topic: str
    # routing/research
    mode: str
    needs_research: bool
    queries: List[str]

    evidence: List[EvidenceItem]
    plan: Optional[Plan]
    sections: Annotated[List[str], operator.add]   # reducer concatenates worker outputs
    final: str


# ====================== LLM ======================
llm = ChatOpenAI(model="gpt-4o-mini")   # Changed to gpt-4o-mini (more stable & cheaper)


# ====================== ROUTER ======================
ROUTER_SYSTEM = """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false): Evergreen topics where correctness does not depend on recent facts.
- hybrid (needs_research=true): Mostly evergreen but needs up-to-date examples/tools/models.
- open_book (needs_research=true): Volatile topics like "this week", "latest", rankings, pricing, etc.

If needs_research=true, output 3–10 high-signal, specific queries.
"""

def router_node(state: State) -> dict:
    topic = state["topic"]
    decider = llm.with_structured_output(RouterDecision)
    
    decision = decider.invoke([
        SystemMessage(content=ROUTER_SYSTEM),
        HumanMessage(content=f"Topic: {topic}"),
    ])

    return {
        "needs_research": decision.needs_research,
        "mode": decision.mode,
        "queries": decision.queries,
    }


def route_next(state: State) -> str:
    return "research" if state.get("needs_research") else "orchestrator"


# ====================== RESEARCH ======================
def _tavily_search(query: str, max_results: int = 5) -> List[dict]:
    tool = TavilySearchResults(max_results=max_results)
    results = tool.invoke({"query": query})

    normalized: List[dict] = []
    for r in results or []:
        normalized.append({
            "title": r.get("title") or "",
            "url": r.get("url") or "",
            "snippet": r.get("content") or r.get("snippet") or "",
            "published_at": r.get("published_date") or r.get("published_at"),
            "source": r.get("source"),
        })
    return normalized


RESEARCH_SYSTEM = """You are a research synthesizer for technical writing.

Given raw web search results, produce a deduplicated list of EvidenceItem objects.

Rules:
- Only include items with a non-empty url.
- Prefer authoritative sources.
- Keep snippets short.
- Deduplicate by URL.
"""

def research_node(state: State) -> dict:
    queries = state.get("queries", [])[:10]  # safety limit
    max_results = 6
    raw_results: List[dict] = []

    for q in queries:
        raw_results.extend(_tavily_search(q, max_results=max_results))

    if not raw_results:
        return {"evidence": []}

    extractor = llm.with_structured_output(EvidencePack)
    pack = extractor.invoke([
        SystemMessage(content=RESEARCH_SYSTEM),
        HumanMessage(content=f"Raw results:\n{raw_results}"),
    ])

    # Deduplicate by URL
    dedup = {e.url: e for e in pack.evidence if e.url}
    return {"evidence": list(dedup.values())}


# ====================== ORCHESTRATOR ======================
def orchestrator(state: State) -> dict:
    planner = llm.with_structured_output(Plan)

    plan = planner.invoke([
        SystemMessage(content="""You are a Substack writer and editor.

Your job is to create a structured writing plan for a Substack article.

The article should:
- Start with a strong hook (personal, curiosity-driven, or contrarian)
- Be conversational and human
- Feel like a story, not a textbook
- Include insights, not just explanations
- Be easy to skim

Return a plan with:
- A compelling blog_title
- audience (specific niche)
- tone
- tasks (each = one section)

Important:
- Use natural, engaging section titles (avoid "Introduction", "Conclusion", etc.)
- Each task must have 3-5 concrete bullets
- Use 'common_mistakes' section type exactly once."""),
        HumanMessage(content=f"Topic: {state['topic']}"),
    ])

    return {"plan": plan}


# ====================== FANOUT ======================
def fanout(state: State):
    return [
        Send(
            "worker",
            {
                "task": task.model_dump(),        # dict
                "topic": state["topic"],
                "mode": state["mode"],
                "plan": state["plan"].model_dump(),
                "evidence": [e.model_dump() for e in state.get("evidence", [])],
            }
        )
        for task in state["plan"].tasks
    ]


# ====================== WORKER (FIXED) ======================
def worker(payload: dict) -> dict:
    task_dict = payload["task"]      # This is now a dict
    topic = payload["topic"]
    plan_dict = payload["plan"]

    # Convert bullets list to text
    bullets_text = "\n- " + "\n- ".join(task_dict["bullets"])

    section_md = llm.invoke([
        SystemMessage(
            content=f"""You are a top Substack writer.

Write **one section** of a Substack article.

Context:
- Topic: {topic}
- Section title: {task_dict['title']}
- Goal: {task_dict['goal']}
- Key ideas:
{bullets_text}

Writing rules:
- Write like a human, not AI
- Short paragraphs (1–3 lines)
- Conversational and slightly opinionated
- No bullet points in final output
- Start strong, end with smooth transition

Output only clean Markdown for this section."""
        ),
        HumanMessage(
            content=(
                f"Blog Title: {plan_dict['blog_title']}\n"
                f"Audience: {plan_dict['audience']}\n"
                f"Tone: {plan_dict['tone']}\n"
                f"Topic: {topic}\n\n"
                f"Section: {task_dict['title']}\n"
                f"Section type: {task_dict['section_type']}\n"
                f"Goal: {task_dict['goal']}\n"
                f"Target words: {task_dict['target_words']}\n"
                f"Bullets:{bullets_text}\n"
            )
        ),
    ]).content.strip()

    return {"sections": [section_md]}


# ====================== REDUCER ======================
def reducer(state: State) -> dict:
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()

    intro = f"*{state['plan'].audience}*\n\n"

    outro = """
---

If you enjoyed this, consider subscribing.

I write about ideas like this regularly—simple, practical, and worth thinking about.
"""

    final_md = f"# {title}\n\n{intro}{body}\n\n{outro}"
    return {"final": final_md}


# ====================== BUILD GRAPH ======================
g = StateGraph(State)

g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("reducer", reducer)

g.add_edge(START, "router")
g.add_conditional_edges("router", route_next, {"research": "research", "orchestrator": "orchestrator"})
g.add_edge("research", "orchestrator")

g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")          # Note: All workers go to reducer
g.add_edge("reducer", END)

app = g.compile()


import datetime

def run(topic: str, save: bool = True):
    initial_state = {
        "topic": topic,
        "mode": "",
        "needs_research": False,
        "queries": [],
        "evidence": [],
        "plan": None,
        "sections": [],
        "final": "",
    }
    
    print("Generating blog... This may take 30-60 seconds.")
    out = app.invoke(initial_state)
    
    final_md = out.get("final", "")
    
    if not final_md:
        print("❌ Failed to generate content.")
        return out
    
    if save:
        # Create nice filename with date
        date_str = datetime.datetime.now().strftime("%Y%m%d")
        safe_topic = "".join(c if c.isalnum() else "_" for c in topic.lower()[:40])
        filename = f"{date_str}_{safe_topic}.md"
        
        with open(filename, "w", encoding="utf-8") as f:
            f.write(final_md)
        
        print(f"✅ Successfully saved to: **{filename}**")
        print(f"Total length: {len(final_md):,} characters")
    
    return out


result = run("State of Multimodal LLMs in 2026")
print(result["final"])

Generating blog... This may take 30-60 seconds.


C:\Users\lamic\AppData\Local\Temp\ipykernel_6724\1551489109.py:105: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=max_results)


✅ Successfully saved to: **20260407_state_of_multimodal_llms_in_2026.md**
Total length: 13,436 characters
# The Future is Here: Multimodal LLMs in 2026

*AI enthusiasts and tech industry professionals*

## A Glimpse into the Future

Imagine a world where your computer doesn't just process text but actually understands the full palette of human expression—images, audio, and even video. By 2026, multimodal LLMs will be seamlessly integrating all these forms of communication into one cohesive experience. Gone will be the days where you have to text an elaborate problem just to get a simple answer. You could just show your device a chaotic mess of papers alongside your facial expressions and sound out your frustrations. It will know exactly how to respond.

But let’s take a moment to consider this: What if your computer could understand your emotions in addition to your words? Picture this—a tough day at work. You're sitting at your desk, feeling overwhelmed. You pull up your virtual assis